Goal: Implement exp3 and plots

# Required Libraries

In [ ]:

import os, json, time, random
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import List, Dict, Tuple, Optional
import torch
import torch.nn as nn
import torch.optim as optim
import torch.backends.cudnn as cudnn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, roc_auc_score, average_precision_score,
    precision_recall_fscore_support, confusion_matrix,precision_recall_curve, auc,precision_score, recall_score, f1_score
)
from tqdm import tqdm as _tqdm
import torch.nn.functional as F
try:
    import timm  # DeiT model
except ImportError as e:
    raise SystemExit("timm is required. Install via `pip install timm`.") from e
import re
from itertools import combinations
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon
import pandas as pd, numpy as np
from functools import reduce
import seaborn as sns
import shutil
import pandas as pd
import numpy as np
import warnings


from sklearn.metrics import confusion_matrix
warnings.filterwarnings('ignore')

cudnn.benchmark = False
cudnn.deterministic = True
torch.set_num_threads(4)

# Functions

In [ ]:
# Defaults 
DEFAULT_SEEDS = [0, 42, 123]
DEFAULT_EPOCHS = 50
DEFAULT_BATCH_SIZE = 64
DEFAULT_BASE_LR = 5e-4        # DeiT paper base LR (scale by batch/256) 
DEFAULT_WD = 5e-2             # 0.05 is common in DeiT/timm recipes 
DEFAULT_WARMUP_EPOCHS = 5
DEFAULT_IMG_SIZE = 224
DEFAULT_NUM_WORKERS = 0
DEFAULT_OPTIM = "adamw"

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


# Utils
def _merge_logits_for_loss_and_pred(logits):
    # Distilled DeiT returns a tuple (cls_logits, dist_logits); non-distilled returns a Tensor
    if isinstance(logits, tuple):
        cls, dist = logits
        merged = 0.5 * (cls + dist)  # simple average for training/inference
        return merged, merged
    return logits, logits


def set_seed(seed: int) -> None:
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    cudnn.deterministic = True; cudnn.benchmark = False

def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

def device_str():
    return "cuda" if torch.cuda.is_available() else "cpu"

@dataclass
class RunConfig:
    model_name: str
    pipeline: str
    fold_name: str
    seed: int
    epochs: int = DEFAULT_EPOCHS
    batch_size: int = DEFAULT_BATCH_SIZE
    base_lr: float = DEFAULT_BASE_LR
    weight_decay: float = DEFAULT_WD
    warmup_epochs: int = DEFAULT_WARMUP_EPOCHS
    img_size: int = DEFAULT_IMG_SIZE
    num_workers: int = DEFAULT_NUM_WORKERS
    optim: str = DEFAULT_OPTIM
    drop_path_rate: float = 0.1      # mild regularization typical for DeiT/timm



# Data
def build_transforms(img_size: int):
    # Keep aug light to match ResNet runs (flip + mild color jitter)
    train_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomApply([transforms.ColorJitter(0.1, 0.1, 0.1, 0.02)], p=0.3),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])
    return train_tf, eval_tf

def build_dataloaders_fold(fold_path: Path, img_size: int, batch_size: int, num_workers: int):
    train_dir, val_dir = fold_path / "train", fold_path / "val"
    if not train_dir.exists() or not val_dir.exists():
        raise FileNotFoundError(f"Missing train/ or val/ under {fold_path}")
    train_tf, eval_tf = build_transforms(img_size)
    train_ds = datasets.ImageFolder(train_dir, transform=train_tf)
    val_ds   = datasets.ImageFolder(val_dir,   transform=eval_tf)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True)
    return train_loader, val_loader, train_ds.classes



# Model & Recipe (DeiT-S)
def build_deit_small_distilled(num_classes: int, drop_path_rate: float = 0.1):
    # timm distilled DeiT-S/16 has an extra distillation head
    # model id: "deit_small_distilled_patch16_224"
    model = timm.create_model(
        "deit_small_distilled_patch16_224",
        pretrained=True,
        num_classes=num_classes,
        drop_path_rate=drop_path_rate,
    )
    return model


def build_deit_small(num_classes: int, drop_path_rate: float = 0.1):
    """
    Non-distilled DeiT-S/16 (fair vs ResNet-18; distilled uses extra teacher).
    """
    model = timm.create_model(
        "deit_small_patch16_224",      # non-distilled variant
        pretrained=True,
        num_classes=num_classes,
        drop_path_rate=drop_path_rate, # mild stochastic depth
    )
    return model


def build_teacher(num_classes: int, device: str):
    teacher = timm.create_model("deit_base_patch16_224", pretrained=False, num_classes=num_classes)
    # try load local
    ckpt = os.path.expanduser("~/.cache/torch/hub/checkpoints/deit_base_patch16_224-a8d89c12.pth")
    if os.path.exists(ckpt):
        state_dict = torch.load(ckpt, map_location="cpu")
        teacher.load_state_dict(state_dict)
        print(f">> Loaded teacher from local {ckpt}")
    else:
        # fallback to pretrained: this will attempt download
        teacher = timm.create_model("deit_base_patch16_224", pretrained=True, num_classes=num_classes)
    teacher.to(device)
    for p in teacher.parameters():
        p.requires_grad = False
    teacher.eval()
    return teacher



def make_optimizer(model: nn.Module, cfg: RunConfig):
    params = [p for p in model.parameters() if p.requires_grad]
    if cfg.optim.lower() == "adamw":
        return optim.AdamW(params, lr=scaled_lr(cfg), weight_decay=cfg.weight_decay)
    raise ValueError(f"Unknown optimizer {cfg.optim}")

def scaled_lr(cfg: RunConfig) -> float:
    # Scale base LR by (batch / 256), common ViT/DeiT practice
    return cfg.base_lr * (cfg.batch_size / 256.0)

def make_warmup_cosine_scheduler(optimizer, cfg: RunConfig, steps_per_epoch: int):
    """
    Simple linear warmup for warmup_epochs, then cosine to 0.
    """
    total_steps   = cfg.epochs * steps_per_epoch
    warmup_steps  = cfg.warmup_epochs * steps_per_epoch
    def lr_lambda(step):
        if step < warmup_steps and warmup_steps > 0:
            return float(step + 1) / float(max(1, warmup_steps))
        # cosine phase
        progress = (step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return 0.5 * (1.0 + np.cos(np.pi * progress))
    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)


# Train / Eval
def train_one_epoch(model, loader, device, criterion, optimizer,
                    use_distillation=False, teacher_model=None, alpha=0.5, T=3.0):
    """
    If use_distillation=True, combine CE loss with KL divergence to teacher.
    Otherwise identical to the original implementation.
    """
    model.train()
    losses, correct, total = [], 0, 0
    t0 = time.perf_counter()

    if use_distillation:
        assert teacher_model is not None, "use_distillation=True but teacher_model=None"
        teacher_model.eval()
        ce_loss_fn = nn.CrossEntropyLoss()
        kl_loss_fn = nn.KLDivLoss(reduction="batchmean")

    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        logits = model(images)
        loss_logits, pred_logits = _merge_logits_for_loss_and_pred(logits)

        if use_distillation:
            # Student outputs
            cls_logits = loss_logits
            with torch.no_grad():
                teacher_logits = teacher_model(images)
                teacher_soft = F.softmax(teacher_logits / T, dim=1)

            loss_ce = ce_loss_fn(cls_logits, targets)
            loss_kd = kl_loss_fn(F.log_softmax(cls_logits / T, dim=1), teacher_soft) * (T * T)
            loss = alpha * loss_ce + (1 - alpha) * loss_kd
        else:
            loss = criterion(loss_logits, targets)

        loss.backward()
        optimizer.step()

        losses.append(loss.item())
        with torch.no_grad():
            preds = pred_logits.argmax(dim=1)
            correct += (preds == targets).sum().item()
            total += targets.size(0)

    t1 = time.perf_counter()
    return float(np.mean(losses)), correct / max(1, total), (t1 - t0), total


@torch.no_grad()
def evaluate(model, loader, device, criterion, class_names: List[str]):
    model.eval()
    losses, all_targets, all_probs, all_preds, all_filenames = [], [], [], [], []
    t0 = time.perf_counter()
    total = 0

    # Get the list of filenames in the same order as the DataLoader
    dataset_samples = loader.dataset.samples  # list of (path, label)
    sample_idx = 0
    for images, targets in loader:
        batch_size = targets.size(0)
        batch_filenames = [os.path.basename(dataset_samples[sample_idx + i][0]) for i in range(batch_size)]
        sample_idx += batch_size
        all_filenames.extend(batch_filenames)


        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        logits = model(images)
        loss_logits, pred_logits = _merge_logits_for_loss_and_pred(logits)
        loss = criterion(loss_logits, targets)
        losses.append(loss.item())
        probs = torch.softmax(pred_logits, dim=1)
        preds = probs.argmax(dim=1)
        all_targets.append(targets.cpu().numpy())
        all_probs.append(probs.cpu().numpy())
        all_preds.append(preds.cpu().numpy())
        total += targets.size(0)
    t1 = time.perf_counter()

    y_true = np.concatenate(all_targets); y_prob = np.concatenate(all_probs); y_pred = np.concatenate(all_preds)
    filenames = np.array(all_filenames)
    acc = accuracy_score(y_true, y_pred); bal_acc = balanced_accuracy_score(y_true, y_pred)

    metrics = {"loss": float(np.mean(losses)), "accuracy": float(acc), "balanced_accuracy": float(bal_acc)}
    if len(class_names) == 2:
        try:    pos_idx = class_names.index("defective")
        except: pos_idx = 1
        try:    auroc = roc_auc_score((y_true == pos_idx).astype(int), y_prob[:, pos_idx])
        except: auroc = np.nan
        try:    auprc = average_precision_score((y_true == pos_idx).astype(int), y_prob[:, pos_idx])
        except: auprc = np.nan
        prec, rec, f1, _ = precision_recall_fscore_support((y_true == pos_idx), (y_pred == pos_idx),
                                                           average="binary", zero_division=0)
        cm = confusion_matrix(y_true, y_pred, labels=[1-pos_idx, pos_idx])
        metrics.update({
            "auroc": float(auroc), "auprc": float(auprc),
            "precision": float(prec), "recall": float(rec), "f1": float(f1),
            "tn": int(cm[0,0]) if cm.shape==(2,2) else np.nan,
            "fp": int(cm[0,1]) if cm.shape==(2,2) else np.nan,
            "fn": int(cm[1,0]) if cm.shape==(2,2) else np.nan,
            "tp": int(cm[1,1]) if cm.shape==(2,2) else np.nan,
        })
    return metrics, y_true, y_pred, y_prob, filenames, (t1 - t0), total



# Runner
def discover_folds(pipeline_root: Path) -> List[Path]:
    folds_dir = pipeline_root / "folds"
    if not folds_dir.exists():
        raise FileNotFoundError(f"{pipeline_root} must contain a 'folds/' directory")
    folds = [d for d in sorted(folds_dir.iterdir()) if d.is_dir() and d.name.startswith("fold_")]
    if not folds:
        raise FileNotFoundError(f"No fold_* directories found under {folds_dir}")
    return folds

def run_deit_on_cvset(
    pipeline_root: str,
    out_dir: str,
    seeds: List[int] = DEFAULT_SEEDS,
    epochs: int = DEFAULT_EPOCHS,
    batch_size: int = DEFAULT_BATCH_SIZE,
    base_lr: float = DEFAULT_BASE_LR,
    weight_decay: float = DEFAULT_WD,
    warmup_epochs: int = DEFAULT_WARMUP_EPOCHS,
    img_size: int = DEFAULT_IMG_SIZE,
    num_workers: int = DEFAULT_NUM_WORKERS,
    drop_path_rate: float = 0.1,
    device: Optional[str] = None,
    use_distilled: bool = False   
):
    device = device or device_str()
    print("Using device:", device)
    pipeline_root = Path(pipeline_root)
    out_root = Path(out_dir); out_root.mkdir(parents=True, exist_ok=True)
    folds = discover_folds(pipeline_root)

    for fold_path in folds:
        fold_name = fold_path.name
        for seed in seeds:
            set_seed(seed)
            run_dir = out_root / pipeline_root.name / fold_name / f"seed{seed}"
            (run_dir / "artifacts").mkdir(parents=True, exist_ok=True)

            # Data
            train_loader, val_loader, class_names = build_dataloaders_fold(
                fold_path, img_size, batch_size, num_workers
            )
            num_classes = len(class_names)
            # Model
            # Build the model — identical recipe/optimizer/loss regardless of distilled or not
            if use_distilled:
                model_name = "deit_small_distilled_patch16_224"
                model = build_deit_small_distilled(num_classes, drop_path_rate).to(device)
                print(">> Using DeiT-Small (distilled) with DeiT-Base teacher")
                teacher_model = build_teacher(num_classes, device)
            else:
                model_name = "deit_small_patch16_224"
                model = build_deit_small(num_classes, drop_path_rate).to(device)
                teacher_model = None
            n_params = count_params(model)
            # Config logs
            cfg = RunConfig(
                model_name=model_name, pipeline=pipeline_root.name, fold_name=fold_name, seed=seed,
                epochs=epochs, batch_size=batch_size, base_lr=base_lr, weight_decay=weight_decay,
                warmup_epochs=warmup_epochs, img_size=img_size, num_workers=num_workers,
                drop_path_rate=drop_path_rate
            )
            with open(run_dir / "run_config.json", "w") as f:
                json.dump(asdict(cfg), f, indent=2)

            

            # Loss — keep weighted CE as in ResNet to handle imbalance fairly
            # (If later need DeiT-style label smoothing, add label_smoothing=0.1 *instead of* weights.)
            # Compute class weights from the training set
            train_targets = []
            for _, targets in train_loader:
                train_targets.extend(targets.numpy().tolist())
            class_counts = np.bincount(train_targets, minlength=num_classes)
            class_weights = class_counts.max() / np.clip(class_counts, 1, None)
            criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=device))

            # Optimizer & scheduler (AdamW + warmup + cosine)
            optimizer = make_optimizer(model, cfg)
            scheduler = make_warmup_cosine_scheduler(optimizer, cfg, steps_per_epoch=len(train_loader))

            # Record model info early
            model_info = {
                "model_name": model_name,
                "pretrained": True,
                "num_parameters": int(n_params),
                "optimizer": cfg.optim,
                "base_lr": cfg.base_lr,
                "scaled_lr": scaled_lr(cfg),
                "weight_decay": cfg.weight_decay,
                "scheduler": "LambdaLR (linear warmup + cosine)",
                "warmup_epochs": cfg.warmup_epochs,
                "batch_size": cfg.batch_size,
                "img_size": cfg.img_size,
                "drop_path_rate": cfg.drop_path_rate,
                "device": device,
                "use_distilled": use_distilled
            }
            with open(run_dir / "model_info.json", "w") as f:
                json.dump(model_info, f, indent=2)

            # Save class mapping
            class_mapping = {
                "class_names": class_names,
                "positive_class": "defective" if "defective" in class_names else class_names[-1],
                "pos_idx": int(class_names.index("defective")) if "defective" in class_names else 1
            }
            with open(run_dir / "class_mapping.json", "w") as f:
                json.dump(class_mapping, f, indent=2)

            # Data meta for later balance plots
            data_meta = {
                "n_train": sum(len(b[1]) for b in train_loader),
                "n_val":   sum(len(b[1]) for b in val_loader),
                "train_class_counts": {class_names[i]: int(c) for i, c in enumerate(class_counts.tolist())},
            }
            with open(run_dir / "data_meta.json", "w") as f:
                json.dump(data_meta, f, indent=2)

            # Train
            history = []
            best_val = -np.inf; best_epoch = -1
            if device == "cuda":
                torch.cuda.reset_peak_memory_stats()

            global_step = 0
            for epoch in range(epochs):
                tr_loss, tr_acc, tr_sec, tr_seen = train_one_epoch(
                        model, train_loader, device, criterion, optimizer,
                        use_distillation=use_distilled, teacher_model=teacher_model,
                        alpha=0.5, T=3.0
                    )

                scheduler.step()

                val_metrics, y_true, y_pred, y_prob, filenames, va_sec, va_seen = evaluate(model, val_loader, device, criterion, class_names)

                # timings & throughput
                gpu_mem = torch.cuda.max_memory_allocated() if device == "cuda" else None
                row = {
                    "epoch": epoch,
                    "train_loss": tr_loss, "train_acc": tr_acc,
                    **{f"val_{k}": v for k, v in val_metrics.items()},
                    "lr": float(optimizer.param_groups[0]["lr"]),
                    "train_sec": tr_sec, "train_img_per_sec": tr_seen / max(1e-9, tr_sec),
                    "val_sec": va_sec, "val_img_per_sec": va_seen / max(1e-9, va_sec),
                    "cuda_max_mem_bytes": int(gpu_mem) if gpu_mem is not None else None,
                }
                history.append(row)
                _tqdm.write(f"[{pipeline_root.name} {fold_name} seed{seed}] Ep {epoch+1}/{epochs} | "
                            f"TrLoss={tr_loss:.4f} | ValAUPRC={val_metrics.get('auprc', float('nan')):.4f} | "
                            f"LR={row['lr']:.2e}")

                # Early best by balanced accuracy (fallback to accuracy)
                monitor = val_metrics.get("balanced_accuracy", val_metrics.get("accuracy", -np.inf))
                if monitor > best_val:
                    best_val = monitor; best_epoch = epoch
                    torch.save(model.state_dict(), run_dir / "artifacts" / "best_model.pt")
                    with open(run_dir / "artifacts" / "best_epoch.json", "w") as f:
                        json.dump({"best_epoch": int(best_epoch), "best_val": float(best_val)}, f)

            # Save training history
            pd.DataFrame(history).to_csv(run_dir / "training_history.csv", index=False)

            # Evaluate best on val again (treated as held-out for this fold)
            if (run_dir / "artifacts" / "best_model.pt").exists():
                model.load_state_dict(torch.load(run_dir / "artifacts" / "best_model.pt", map_location=device))
            
            val_metrics, y_true, y_pred, y_prob, filenames, _, _ = evaluate(model, val_loader, device, criterion, class_names)

            cm = confusion_matrix(y_true, y_pred)
            pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(run_dir / "val_confusion_matrix.csv")


            preds_df = pd.DataFrame({
                "filename": filenames,
                "y_true": y_true,
                "y_pred": y_pred,
            })
            for c in range(len(class_names)):
                preds_df[f"p_class_{c}"] = y_prob[:, c]
            preds_df.to_csv(run_dir / "val_predictions.csv", index=False)

            with open(run_dir / "val_metrics.json", "w") as f:
                json.dump(val_metrics, f, indent=2)




## non-dist version

In [ ]:
# On the single CV set selected in Exp-2 
SELECTED_CVSET_ROOT = "./blurclahe_data_folds/blurclahe_inner_seed79"  # median perf split

OUT_DIR = "./results_experiment3/DeiT-Small16"

run_deit_on_cvset(
    pipeline_root=SELECTED_CVSET_ROOT,
    out_dir=OUT_DIR,
    seeds=[0, 42, 123],
    epochs=50,
    batch_size=64,
    base_lr=5e-4,         # DeiT base LR (scaled by batch/256 inside) 
    weight_decay=5e-2,    # common for DeiT/timm recipes 
    warmup_epochs=5,
    img_size=224,
    num_workers=0,
    drop_path_rate=0.1,
    device="cuda",
)

Using device: cuda


model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]

c:\Users\myria\miniconda3\envs\tf_gpu\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\myria\.cache\huggingface\hub\models--timm--deit_small_patch16_224.fb_in1k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


[blurclahe_inner_seed79 fold_0 seed0] Ep 1/50 | TrLoss=0.6121 | ValAUPRC=0.9099 | LR=3.13e-06
[blurclahe_inner_seed79 fold_0 seed0] Ep 2/50 | TrLoss=0.5197 | ValAUPRC=0.9751 | LR=4.69e-06
[blurclahe_inner_seed79 fold_0 seed0] Ep 3/50 | TrLoss=0.4126 | ValAUPRC=0.9977 | LR=6.25e-06
[blurclahe_inner_seed79 fold_0 seed0] Ep 4/50 | TrLoss=0.3227 | ValAUPRC=0.9976 | LR=7.81e-06
[blurclahe_inner_seed79 fold_0 seed0] Ep 5/50 | TrLoss=0.2612 | ValAUPRC=0.9991 | LR=9.37e-06
[blurclahe_inner_seed79 fold_0 seed0] Ep 6/50 | TrLoss=0.2395 | ValAUPRC=0.9925 | LR=1.09e-05
[blurclahe_inner_seed79 fold_0 seed0] Ep 7/50 | TrLoss=0.1769 | ValAUPRC=0.9915 | LR=1.25e-05
[blurclahe_inner_seed79 fold_0 seed0] Ep 8/50 | TrLoss=0.1379 | ValAUPRC=0.9900 | LR=1.41e-05
[blurclahe_inner_seed79 fold_0 seed0] Ep 9/50 | TrLoss=0.1115 | ValAUPRC=0.9822 | LR=1.56e-05
[blurclahe_inner_seed79 fold_0 seed0] Ep 10/50 | TrLoss=0.0897 | ValAUPRC=0.9895 | LR=1.72e-05
[blurclahe_inner_seed79 fold_0 seed0] Ep 11/50 | TrLoss=0.0

C:\Users\myria\AppData\Local\Temp\ipykernel_67732\3994099794.py:387: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(run_dir / "artifacts" / "

...

C:\Users\myria\AppData\Local\Temp\ipykernel_67732\3994099794.py:387: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(run_dir / "artifacts" / "

[blurclahe_inner_seed79 fold_4 seed42] Ep 1/50 | TrLoss=0.6326 | ValAUPRC=0.5142 | LR=3.13e-06
[blurclahe_inner_seed79 fold_4 seed42] Ep 2/50 | TrLoss=0.4917 | ValAUPRC=0.5836 | LR=4.69e-06
[blurclahe_inner_seed79 fold_4 seed42] Ep 3/50 | TrLoss=0.3449 | ValAUPRC=0.5932 | LR=6.25e-06
[blurclahe_inner_seed79 fold_4 seed42] Ep 4/50 | TrLoss=0.2374 | ValAUPRC=0.6440 | LR=7.81e-06
[blurclahe_inner_seed79 fold_4 seed42] Ep 5/50 | TrLoss=0.1786 | ValAUPRC=0.6765 | LR=9.37e-06
[blurclahe_inner_seed79 fold_4 seed42] Ep 6/50 | TrLoss=0.1447 | ValAUPRC=0.7235 | LR=1.09e-05
[blurclahe_inner_seed79 fold_4 seed42] Ep 7/50 | TrLoss=0.1253 | ValAUPRC=0.7464 | LR=1.25e-05
[blurclahe_inner_seed79 fold_4 seed42] Ep 8/50 | TrLoss=0.0896 | ValAUPRC=0.7724 | LR=1.41e-05
[blurclahe_inner_seed79 fold_4 seed42] Ep 9/50 | TrLoss=0.0714 | ValAUPRC=0.7881 | LR=1.56e-05
[blurclahe_inner_seed79 fold_4 seed42] Ep 10/50 | TrLoss=0.0685 | ValAUPRC=0.7313 | LR=1.72e-05
[blurclahe_inner_seed79 fold_4 seed42] Ep 11/50 |

C:\Users\myria\AppData\Local\Temp\ipykernel_67732\3994099794.py:387: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(run_dir / "artifacts" / "

[blurclahe_inner_seed79 fold_4 seed123] Ep 1/50 | TrLoss=0.6693 | ValAUPRC=0.3422 | LR=3.13e-06
[blurclahe_inner_seed79 fold_4 seed123] Ep 2/50 | TrLoss=0.5310 | ValAUPRC=0.5353 | LR=4.69e-06
[blurclahe_inner_seed79 fold_4 seed123] Ep 3/50 | TrLoss=0.3823 | ValAUPRC=0.5780 | LR=6.25e-06
[blurclahe_inner_seed79 fold_4 seed123] Ep 4/50 | TrLoss=0.2638 | ValAUPRC=0.6226 | LR=7.81e-06
[blurclahe_inner_seed79 fold_4 seed123] Ep 5/50 | TrLoss=0.1988 | ValAUPRC=0.6643 | LR=9.37e-06
[blurclahe_inner_seed79 fold_4 seed123] Ep 6/50 | TrLoss=0.1523 | ValAUPRC=0.6665 | LR=1.09e-05
[blurclahe_inner_seed79 fold_4 seed123] Ep 7/50 | TrLoss=0.1215 | ValAUPRC=0.6695 | LR=1.25e-05
[blurclahe_inner_seed79 fold_4 seed123] Ep 8/50 | TrLoss=0.1056 | ValAUPRC=0.7073 | LR=1.41e-05
[blurclahe_inner_seed79 fold_4 seed123] Ep 9/50 | TrLoss=0.0667 | ValAUPRC=0.7506 | LR=1.56e-05
[blurclahe_inner_seed79 fold_4 seed123] Ep 10/50 | TrLoss=0.0543 | ValAUPRC=0.7514 | LR=1.72e-05
[blurclahe_inner_seed79 fold_4 seed123]

C:\Users\myria\AppData\Local\Temp\ipykernel_67732\3994099794.py:387: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(run_dir / "artifacts" / "

### TP,TN,FP,FN FOLDERS

In [ ]:
def split_val_errors_to_folders(
    run_dir: Path,
    cv_root: Path,
    fold_name: str,
):

    # Load predictions and mapping
    preds_path = run_dir / "val_predictions.csv"
    mapping = json.load(open(run_dir / "class_mapping.json", "r"))
    class_names = mapping["class_names"]
    pos_idx = mapping["pos_idx"]
    # For two classes, label names:
    neg_idx = 1 - pos_idx

    preds = pd.read_csv(preds_path)

    # Rebuild validation dataset to get file paths
    val_dir = cv_root / "folds" / fold_name / "val"
    # We need the same image size / normalization, but paths come from samples
    # The transform doesn’t matter for us here (just for dataset instantiation)
    eval_tf = transforms.Compose([
        transforms.Resize((224, 224)),  # size should match original but not needed for path
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ])
    val_ds = datasets.ImageFolder(val_dir, transform=eval_tf)
    filepaths = [p for (p, _) in val_ds.samples]

    # Sanity: preds rows should match len(filepaths)
    if len(preds) != len(filepaths):
        raise RuntimeError(f"Length mismatch: preds {len(preds)} vs files {len(filepaths)}")

    # Prepare output dirs
    out_base = run_dir / "val_error_splits"
    for sub in ["tp", "tn", "fp", "fn"]:
        (out_base / sub).mkdir(parents=True, exist_ok=True)

    # Iterate and copy
    for idx, row in preds.iterrows():
        true = int(row["y_true"])
        pred = int(row["y_pred"])
        src = Path(filepaths[idx])
        fname = src.name  # could also include subdir, but name is usually enough

        # Decide folder
        if (true == pos_idx) and (pred == pos_idx):
            sub = "tp"
        elif (true == neg_idx) and (pred == neg_idx):
            sub = "tn"
        elif (true != pos_idx) and (pred == pos_idx):
            # true is negative, but predicted positive
            sub = "fp"
        elif (true == pos_idx) and (pred != pos_idx):
            sub = "fn"
        else:
            # In multiclass or weird mapping, fallback
            sub = None

        if sub is not None:
            dst = out_base / sub / fname
            try:
                # Option 1: hard link (if same filesystem)
                os.link(src, dst)
            except Exception:
                # fallback to copy
                shutil.copy2(src, dst)

    print(f"Done — images split into {out_base}/{{tp,tn,fp,fn}}")


if __name__ == "__main__":
    run_dir = Path("results_experiment3/DeiT-Small16/blurclahe_inner_seed79/fold_4/seed0") # change for each folder you want to include fp,fn,tp,tn
    cv_root = Path("blurclahe_data_folds/blurclahe_inner_seed79")
    fold_name = "fold_4"
    split_val_errors_to_folders(run_dir, cv_root, fold_name)


Done — images split into results_experiment3\DeiT-Small16\blurclahe_inner_seed79\fold_4\seed0\val_error_splits/{tp,tn,fp,fn}


--> resnet results copy paste from exp2 and patchcore results copy paste from ubuntu

### inspcect results

In [ ]:
# CONFIG
BASE = Path("results_experiment3")
RESNET_DIR    = BASE / "Resnet-18"
DEIT_DIR      = BASE / "DeiT-Small16"
PATCHCORE_DIRS = [BASE / "Patchcore"]
SELECTED_CVSET_NAME = "blurclahe_inner_seed79"   # Exp2 chosen split

OUT_DIR = BASE / "_analysis"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PRIMARY_METRIC = "auprc"
BASELINE_MODEL_FOR_DELTAS = "Resnet-18"


# HELPERS
def bootstrap_ci(x, n_boot=4000, alpha=0.05, seed=123):
    rng = np.random.default_rng(seed)
    x = np.array(x, dtype=float)
    if len(x) == 0: return np.nan, np.nan, np.nan, np.nan
    boots = [rng.choice(x, size=len(x), replace=True).mean() for _ in range(n_boot)]
    lo = np.percentile(boots, 100*alpha/2)
    hi = np.percentile(boots, 100*(1-alpha/2))
    return float(x.mean()), float(lo), float(hi), float(x.std(ddof=1))

def _safe_json_load(path):
    try:
        with open(path, "r") as f: return json.load(f)
    except Exception: return {}

def _digits(s):
    m = re.search(r"(\d+)$", str(s))
    return int(m.group(1)) if m else None

def normalize_fold_idx_from_name(name: str) -> int:
    """
    Map 'fold_079', 'fold_179', 'mvtec_fold_379' -> 0..4.
    If it's already 'fold_0'..'fold_4', return that integer.
    Raise ValueError for non-fold labels (e.g., 'Mean').
    """
    s = str(name)
    m = re.search(r"(?:^|_)fold_(\d+)", s) or re.search(r"mvtec_fold_(\d+)", s)
    if not m:
        m = re.search(r"(\d+)$", s)
        if not m:
            raise ValueError(f"Cannot parse fold index from: {name}")
    num = m.group(1)  # '079', '179', '0', etc.
    if num.endswith("79") and len(num) >= 3:
        return int(num[0])  # '079' -> 0, '179' -> 1, ...
    return int(num)

def _find_val_metrics(root: Path):
    """Yield (fold_idx, seed, path_to_val_metrics.json, model_name, cvset)."""
    for cvset_dir in sorted([d for d in root.iterdir() if d.is_dir()]):
        cvset = cvset_dir.name
        for fold_dir in sorted(cvset_dir.glob("fold_*")):
            fidx = _digits(fold_dir.name)
            for seed_dir in sorted(fold_dir.glob("seed*")):
                s = _digits(seed_dir.name)
                p = seed_dir / "val_metrics.json"
                if p.exists():
                    yield fidx, s, p, root.name, cvset

def _seed_from_runname(name: str):
    m = re.match(r"seed(\d+)_", name, flags=re.IGNORECASE)
    if m: return int(m.group(1))
    m = re.search(r"_S(\d+)(?:$|_)", name)
    if m: return int(m.group(1))
    return None  # unknown -> will set to -1 later

# LOAD RESNET + DEIT
rows = []

for root in [RESNET_DIR, DEIT_DIR]:
    if not root.exists():
        print(f"[WARN] Missing: {root}")
        continue
    for fidx, seed, mpath, model_name, cvset in _find_val_metrics(root):
        m = _safe_json_load(mpath)
        rows.append({
            "model": model_name,
            "cvset": cvset,
            "fold": int(fidx),
            "seed": int(seed),
            "auprc": m.get("auprc", np.nan),
            "auroc": m.get("auroc", np.nan),
            "balanced_accuracy": m.get("balanced_accuracy", np.nan),
            "precision": m.get("precision", np.nan),
            "recall": m.get("recall", np.nan),
            "f1": m.get("f1", np.nan),
            "source": str(mpath.relative_to(BASE)),
        })


# LOAD PATCHCORE
# collect all run dirs from either capitalization
pc_run_dirs = []
for d in PATCHCORE_DIRS:
    if d.exists():
        pc_run_dirs.extend([p for p in d.iterdir() if p.is_dir()])

for run_dir in sorted(pc_run_dirs):
    seed = _seed_from_runname(run_dir.name)

    #summary CSV: use results.csv if present; ignore 'Mean' row
    best_thr = {}
    summary_csv = None
    for p in run_dir.glob("*.csv"):
        if p.name.lower() in {"results.csv", "summary.csv"}:
            summary_csv = p
            break
    if summary_csv:
        try:
            s = pd.read_csv(summary_csv)
            rn_col  = next((c for c in s.columns if c.lower().startswith("row")), None)
            thr_col = next((c for c in s.columns if "threshold" in c.lower()), None)
            if rn_col and thr_col:
                for _, r in s.iterrows():
                    name = str(r[rn_col]).strip()
                    try:
                        fold_idx = normalize_fold_idx_from_name(name)
                    except Exception:
                        # non-fold line like 'Mean' -> ignore
                        continue
                    try:
                        best_thr[int(fold_idx)] = float(r[thr_col])
                    except Exception:
                        pass
        except Exception as e:
            print(f"[WARN] Could not parse summary: {summary_csv} ({e})")

    # -- per-image CSVs
    per_image_root = run_dir / "per_image"
    for fold_dir in sorted(per_image_root.glob("*")):
        try:
            fidx = normalize_fold_idx_from_name(fold_dir.name)
        except Exception:
            continue
        csvs = list(fold_dir.glob("*.csv"))
        if not csvs:
            continue
        dfi = pd.read_csv(csvs[0])

        y_true = dfi["is_defect"].astype(int).values
        scores = dfi["score"].astype(float).values

        if "predicted_label" in dfi.columns:
            pl = dfi["predicted_label"]
            if pl.dtype == bool:
                y_hat = pl.astype(int).values
            else:
                y_hat = (pl.astype(str).str.lower()
                           .map({"1":1,"true":1,"0":0,"false":0})
                           .fillna(pl).astype(int).values)
        else:
            thr = best_thr.get(fidx, 0.5)
            y_hat = (scores >= thr).astype(int)

        # metrics
        try: auprc = average_precision_score(y_true, scores)
        except Exception: auprc = np.nan
        try: auroc = roc_auc_score(y_true, scores)
        except Exception: auroc = np.nan
        try: bal_acc = balanced_accuracy_score(y_true, y_hat)
        except Exception: bal_acc = np.nan

        tn, fp, fn, tp = confusion_matrix(y_true, y_hat, labels=[0,1]).ravel()
        prec = tp / max(1, (tp+fp))
        rec  = tp / max(1, (tp+fn))
        f1   = 2*prec*rec / max(1e-9, (prec+rec))

        rows.append({
            "model": "PatchCore",
            "cvset": SELECTED_CVSET_NAME,   # normalize to chosen CV set
            "fold": int(fidx),
            "seed": int(seed) if seed is not None else -1,
            "auprc": float(auprc),
            "auroc": float(auroc),
            "balanced_accuracy": float(bal_acc),
            "precision": float(prec),
            "recall": float(rec),
            "f1": float(f1),
            "threshold_used": best_thr.get(fidx, None),
            "source": str(csvs[0].relative_to(BASE)),
        })


# BUILD TIDY TABLE
df = pd.DataFrame(rows)
if df.empty:
    raise SystemExit("No results loaded. Check folder paths.")

df["fold"] = pd.to_numeric(df["fold"], errors="coerce")
df["seed"] = pd.to_numeric(df["seed"], errors="coerce")

tidy_path = OUT_DIR / "exp3_tidy_metrics.csv"
df.to_csv(tidy_path, index=False)
print("Saved:", tidy_path)


# SUMMARY + WILCOXON
def summarize_by_model(df, metric=PRIMARY_METRIC):
    out = []
    for m in sorted(df["model"].unique()):
        x = df[df["model"]==m][metric].dropna().values
        if len(x)==0: continue
        mean, lo, hi, sd = bootstrap_ci(x, n_boot=4000, alpha=0.05, seed=123)
        out.append({"model": m, f"{metric}_mean": mean, f"{metric}_ci_lo": lo,
                    f"{metric}_ci_hi": hi, f"{metric}_sd": sd, "n": len(x)})
    return pd.DataFrame(out).sort_values(f"{metric}_mean", ascending=False)

summ = summarize_by_model(df, PRIMARY_METRIC)
summ.to_csv(OUT_DIR / f"summary_{PRIMARY_METRIC}.csv", index=False)
print("Saved:", OUT_DIR / f"summary_{PRIMARY_METRIC}.csv")
print("\n== Summary ({} mean ± 95% CI) ==\n".format(PRIMARY_METRIC), summ.to_string(index=False))

def paired_wilcoxon(df, mA, mB, metric=PRIMARY_METRIC):
    A = df[df["model"]==mA][["fold","seed",metric]].dropna()
    B = df[df["model"]==mB][["fold","seed",metric]].dropna()
    merged = pd.merge(A, B, on=["fold","seed"], suffixes=("_A","_B"))
    if len(merged)==0:
        return {"A":mA, "B":mB, "n_pairs":0, "delta_mean":np.nan, "p_wilcoxon":np.nan}
    delta = merged[f"{metric}_A"] - merged[f"{metric}_B"]
    try: _, p = wilcoxon(delta)
    except ValueError: p = np.nan
    return {"A":mA, "B":mB, "n_pairs":int(len(delta)), "delta_mean":float(delta.mean()), "p_wilcoxon":float(p)}

pairs = []
mods = sorted(df["model"].unique())
for A, B in combinations(mods, 2):
    pairs.append(paired_wilcoxon(df, A, B, PRIMARY_METRIC))
pd.DataFrame(pairs).to_csv(OUT_DIR / f"paired_wilcoxon_{PRIMARY_METRIC}.csv", index=False)
print("Saved:", OUT_DIR / f"paired_wilcoxon_{PRIMARY_METRIC}.csv")


# PLOTS
# Bar ± CI
plt.figure(figsize=(8,4.5))
x = np.arange(len(summ))
y = summ[f"{PRIMARY_METRIC}_mean"].values
yerr = np.vstack([y - summ[f"{PRIMARY_METRIC}_ci_lo"].values,
                  summ[f"{PRIMARY_METRIC}_ci_hi"].values - y])
plt.bar(x, y, yerr=yerr, capsize=4)
plt.xticks(x, summ["model"], rotation=15)
plt.ylabel(PRIMARY_METRIC.upper())
plt.title(f"Exp3: {PRIMARY_METRIC.upper()} mean ± 95% CI by model")
plt.tight_layout()
plt.savefig(OUT_DIR / f"bar_{PRIMARY_METRIC}_mean_ci.png", dpi=220)
plt.close()

# Violin
plt.figure(figsize=(8,4.5))
data = [df[df["model"]==m][PRIMARY_METRIC].dropna().values for m in mods]
plt.violinplot(data, showmeans=True, showextrema=False)
plt.xticks(np.arange(1,len(mods)+1), mods, rotation=15)
plt.ylabel(PRIMARY_METRIC.upper())
plt.title(f"Exp3: {PRIMARY_METRIC.upper()} across fold×seed runs")
plt.tight_layout()
plt.savefig(OUT_DIR / f"violin_{PRIMARY_METRIC}_by_model.png", dpi=220)
plt.close()

# Fold-wise heatmap (mean across seeds)
pivot = (df.groupby(["fold","model"])[PRIMARY_METRIC].mean().unstack("model"))
if pivot.shape[0] > 0:
    plt.figure(figsize=(8,5))
    im = plt.imshow(pivot.values, aspect="auto", cmap="viridis", vmin=0, vmax=1)
    plt.colorbar(im, label=f"Mean {PRIMARY_METRIC.upper()} (across seeds)")
    plt.yticks(np.arange(len(pivot.index)), [f"fold_{i}" for i in pivot.index])
    plt.xticks(np.arange(len(pivot.columns)), list(pivot.columns), rotation=15)
    plt.title(f"Exp3: Fold-wise mean {PRIMARY_METRIC.upper()} per model")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"heatmap_{PRIMARY_METRIC}_fold_by_model.png", dpi=220)
    plt.close()

# Estimation deltas vs baseline (paired by fold×seed)
if BASELINE_MODEL_FOR_DELTAS in mods:
    baseline = BASELINE_MODEL_FOR_DELTAS
    others = [m for m in mods if m != baseline]
    plt.figure(figsize=(9,5))
    ypos, labels = [], []
    y0 = 0
    for m in others:
        A = df[df["model"]==m][["fold","seed",PRIMARY_METRIC]].dropna()
        B = df[df["model"]==baseline][["fold","seed",PRIMARY_METRIC]].dropna()
        merged = pd.merge(A, B, on=["fold","seed"], suffixes=("_A","_B"))
        if len(merged)==0: continue
        delta = merged[f"{PRIMARY_METRIC}_A"] - merged[f"{PRIMARY_METRIC}_B"]
        plt.scatter(delta, np.full_like(delta, y0, dtype=float), alpha=0.75)
        dmean, dlo, dhi, _ = bootstrap_ci(delta.values, n_boot=4000, alpha=0.05, seed=123)
        plt.hlines(y0, dlo, dhi, lw=6)
        plt.plot([dmean], [y0], "s")
        labels.append(f"{m} − {baseline}"); ypos.append(y0); y0 += 1
    plt.axvline(0, ls="--", lw=1, color="tab:blue")
    plt.yticks(ypos, labels)
    plt.xlabel(f"{PRIMARY_METRIC.upper()} difference (paired by fold×seed)")
    plt.title(f"Exp3: Estimation plot of paired deltas vs. {baseline}")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"estimation_deltas_vs_{baseline}.png", dpi=220)
    plt.close()

print("\nDone. Check:", OUT_DIR)
print(" - exp3_tidy_metrics.csv, summary_AUPRC.csv, paired_wilcoxon_AUPRC.csv")
print(" - bar/violin/heatmap/estimation plots")


Saved: results_experiment3\_analysis\exp3_tidy_metrics.csv
Saved: results_experiment3\_analysis\summary_auprc.csv

== Summary (auprc mean ± 95% CI) ==
        model  auprc_mean  auprc_ci_lo  auprc_ci_hi  auprc_sd  n
DeiT-Small16    0.910707     0.863265     0.954207  0.095993 15
   Resnet-18    0.861196     0.795837     0.922570  0.133465 15
   PatchCore    0.608118     0.516688     0.694499  0.184592 15
Saved: results_experiment3\_analysis\paired_wilcoxon_auprc.csv

Done. Check: results_experiment3\_analysis
 - exp3_tidy_metrics.csv, summary_AUPRC.csv, paired_wilcoxon_AUPRC.csv
 - bar/violin/heatmap/estimation plots


### overall metrics for best model (deit-s) that is going to be tuned

In [ ]:

# Root where training results are saved
OUT_DIR = Path("results_experiment3/DeiT-Small16")

records = []

# Recursively find all val_metrics.json files
for val_file in OUT_DIR.rglob("val_metrics.json"):
    # Expected: results_experiment3/DeiT-Small16/blurclahe_inner_seed79/fold_X/seedY/val_metrics.json
    parts = val_file.parts
    try:
        seed_dir = val_file.parent
        fold_dir = seed_dir.parent
        cvset_dir = fold_dir.parent

        seed_name = seed_dir.name.replace("seed", "")
        fold_name = fold_dir.name
        cvset_name = cvset_dir.name
    except Exception as e:
        print(f"Skipping unexpected path: {val_file}")
        continue

    with open(val_file) as f:
        m = json.load(f)

    rec = {
        "cvset": cvset_name,
        "fold": fold_name,
        "seed": seed_name,
        "precision": m.get("precision", np.nan),
        "recall": m.get("recall", np.nan),
        "auprc": m.get("auprc", np.nan),
        "f1": m.get("f1", np.nan),
        "tp": m.get("tp", np.nan),
        "tn": m.get("tn", np.nan),
        "fp": m.get("fp", np.nan),
        "fn": m.get("fn", np.nan),
    }
    records.append(rec)

# Create dataframe
df = pd.DataFrame(records)
if df.empty:
    raise SystemExit("No val_metrics.json files found — check OUT_DIR path.")

# Compute mean metrics per fold (across seeds)
fold_means = (
    df.groupby(["cvset", "fold"])
      .agg({k: "mean" for k in ["precision","recall","auprc","f1","tp","tn","fp","fn"]})
      .reset_index()
)
fold_means["seed"] = "mean_across_seeds"

# Compute global mean (across folds & seeds)
overall_mean = {k: df[k].mean() for k in ["precision","recall","auprc","f1","tp","tn","fp","fn"]}
overall_row = {"cvset": df["cvset"].iloc[0], "fold": "ALL", "seed": "ALL", **overall_mean}

# Combine everything
summary = pd.concat([df, fold_means, pd.DataFrame([overall_row])], ignore_index=True)

# Save
out_path = OUT_DIR / "deit-s_overall_metrics.csv"
summary.to_csv(out_path, index=False)
print(f"Summary saved to {out_path}")
print(summary.tail(10))


### enseble metrics

In [ ]:

# SETTINGS 
ROOT = Path("results_experiment3/DeiT-Small16/blurclahe_inner_seed79")  
OUT_DIR = ROOT / "aggregate_results"
OUT_DIR.mkdir(parents=True, exist_ok=True)



#  Helpers 
def load_seed_preds(fold_dir):
    """Load val_predictions.csv from all seeds in this fold."""
    seeds = {}
    for sd in sorted(fold_dir.iterdir()):
        if sd.is_dir() and sd.name.startswith("seed"):
            f = sd / "val_predictions.csv"
            if f.exists():
                seeds[sd.name] = pd.read_csv(f)
    return seeds


def ensemble_probs(seeds_dict):
    """Average probabilities across seeds (same filenames)."""
    merged = reduce(lambda l, r: pd.merge(l, r, on=["filename","y_true"], suffixes=("","_r")), seeds_dict.values())
    pcols = [c for c in merged.columns if c.startswith("p_class_")]
    groups = {}
    for c in pcols:
        base = c.split("_r")[0]
        groups.setdefault(base, []).append(c)
    for base, cols in groups.items():
        merged[base+"_avg"] = merged[cols].mean(axis=1)
    avg_cols = [f"{base}_avg" for base in groups.keys()]
    P = merged[avg_cols].to_numpy()
    merged["y_pred_ensemble"] = P.argmax(axis=1)
    return merged[["filename","y_true","y_pred_ensemble"] + avg_cols]


def extract_tp_tn_fp_fn(y_true, y_pred):
    """Compute TP, TN, FP, FN assuming class 0 = positive ('defective')."""
    cm = confusion_matrix(y_true, y_pred, labels=[0,1])
    if cm.shape != (2,2):
        return dict(TP=0, TN=0, FP=0, FN=0)
    tp = int(cm[0,0])  # true 0 pred 0
    fn = int(cm[0,1])  # true 0 pred 1
    fp = int(cm[1,0])  # true 1 pred 0
    tn = int(cm[1,1])  # true 1 pred 1
    return dict(TP=tp, TN=tn, FP=fp, FN=fn)


def compute_metrics(y_true, y_pred, y_prob=None):
    """Return common metrics assuming class 0 = positive."""
    metrics = dict(
        acc = accuracy_score(y_true, y_pred),
        bal_acc = balanced_accuracy_score(y_true, y_pred),
        prec = precision_score(y_true, y_pred, pos_label=0, zero_division=0),
        rec  = recall_score(y_true, y_pred, pos_label=0, zero_division=0),
        f1   = f1_score(y_true, y_pred, pos_label=0, zero_division=0),
        auprc = np.nan
    )
    if y_prob is not None:
        try:
            metrics["auprc"] = average_precision_score((np.array(y_true)==0).astype(int), np.array(y_prob))
        except Exception:
            metrics["auprc"] = np.nan
    return metrics


def average_confusion_across_seeds(seeds_dict):
    """Compute per-seed confusion/metrics and average them."""
    stats_list, metrics_list = [], []
    for df in seeds_dict.values():
        s = extract_tp_tn_fp_fn(df["y_true"], df["y_pred"])
        m = compute_metrics(df["y_true"], df["y_pred"], y_prob=df["p_class_0"])
        stats_list.append(s)
        metrics_list.append(m)
    stats_avg = {k: np.mean([d[k] for d in stats_list]) for k in stats_list[0]}
    metrics_avg = {k: np.mean([d[k] for d in metrics_list]) for k in metrics_list[0]}
    return stats_avg, metrics_avg


def print_cm_and_metrics(name, stats, metrics):
    """Pretty print confusion matrix + metrics."""
    tp, tn, fp, fn = stats["TP"], stats["TN"], stats["FP"], stats["FN"]
    print(f"\n{name}")
    print("Confusion matrix (class 0 = defective positive):")
    print(f"TP={tp:5.1f} | FN={fn:5.1f}")
    print(f"FP={fp:5.1f} | TN={tn:5.1f}")
    print(f"Total={tp+tn+fp+fn:5.1f}")
    print("Metrics: "
          f"Acc={metrics['acc']:.3f}, "
          f"BalAcc={metrics['bal_acc']:.3f}, "
          f"Prec={metrics['prec']:.3f}, "
          f"Rec={metrics['rec']:.3f}, "
          f"F1={metrics['f1']:.3f}, "
          f"AUPRC={metrics['auprc']:.3f}")


#  Main
def compute_fold_confusions(root):
    folds = sorted([d for d in root.iterdir() if d.is_dir() and d.name.startswith("fold_")])
    all_true_ens, all_pred_ens, all_prob_ens = [], [], []
    all_true_avg, all_pred_avg, all_prob_avg = [], [], []

    results_summary = []

    for fold_dir in folds:
        seed_preds = load_seed_preds(fold_dir)
        if not seed_preds:
            continue

        # (1) Average across seeds (metrics)
        stats_avg, metrics_avg = average_confusion_across_seeds(seed_preds)
        for df in seed_preds.values():
            all_true_avg.extend(df["y_true"].to_list())
            all_pred_avg.extend(df["y_pred"].to_list())
            all_prob_avg.extend(df["p_class_0"].to_list())

        # (2) Ensemble avg probabilities across seeds
        df_ens = ensemble_probs(seed_preds)
        stats_ens = extract_tp_tn_fp_fn(df_ens["y_true"], df_ens["y_pred_ensemble"])
        metrics_ens = compute_metrics(df_ens["y_true"], df_ens["y_pred_ensemble"], y_prob=df_ens["p_class_0_avg"])
        all_true_ens.extend(df_ens["y_true"].to_list())
        all_pred_ens.extend(df_ens["y_pred_ensemble"].to_list())
        all_prob_ens.extend(df_ens["p_class_0_avg"].to_list())

        # Save ensemble predictions in a new folder
        ens_dir = fold_dir / "ensemble_predictions"
        ens_dir.mkdir(exist_ok=True)
        df_ens.to_csv(ens_dir / "val_predictions_ensemble.csv", index=False)

        # Also save summary confusion tables
        pd.DataFrame([stats_avg]).to_csv(fold_dir/"cm_avg_across_seeds.csv", index=False)
        pd.DataFrame([stats_ens]).to_csv(fold_dir/"cm_ensemble_avg_probs.csv", index=False)

        # Print per fold
        print(f"\n- {fold_dir.name.upper()} -")
        print_cm_and_metrics("No Ensemble (average over seeds)", stats_avg, metrics_avg)
        print_cm_and_metrics("Ensemble (average probabilities)", stats_ens, metrics_ens)

        results_summary.append({
            "fold": fold_dir.name,
            **{f"{k}_avg": v for k,v in stats_avg.items()},
            **{f"{k}_ens": v for k,v in stats_ens.items()},
            "auprc_avg": metrics_avg["auprc"],
            "auprc_ens": metrics_ens["auprc"]
        })

    # overall across folds (ensemble and averaged) 
    overall_avg_stats = {k: np.sum([r[f"{k}_avg"] for r in results_summary]) for k in ["TP","TN","FP","FN"]}
    overall_ens_stats = {k: np.sum([r[f"{k}_ens"] for r in results_summary]) for k in ["TP","TN","FP","FN"]}

    def compute_metrics_from_counts(d, y_true, y_pred, y_prob):
        TP, TN, FP, FN = d["TP"], d["TN"], d["FP"], d["FN"]
        total = TP + TN + FP + FN
        acc = (TP + TN) / total if total else np.nan
        tpr = TP / (TP + FN) if (TP + FN) else np.nan
        tnr = TN / (TN + FP) if (TN + FP) else np.nan
        bal_acc = np.nanmean([tpr, tnr])
        prec = TP / (TP + FP) if (TP + FP) else np.nan
        rec = tpr
        f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) else np.nan
        try:
            auprc = average_precision_score((np.array(y_true)==0).astype(int), np.array(y_prob))
        except Exception:
            auprc = np.nan
        return dict(acc=acc, bal_acc=bal_acc, prec=prec, rec=rec, f1=f1, auprc=auprc)

    overall_avg_metrics = compute_metrics_from_counts(overall_avg_stats, all_true_avg, all_pred_avg, all_prob_avg)
    overall_ens_metrics = compute_metrics_from_counts(overall_ens_stats, all_true_ens, all_pred_ens, all_prob_ens)

    pd.DataFrame(results_summary).to_csv(OUT_DIR/"per_fold_summary.csv", index=False)
    pd.DataFrame([overall_avg_stats | overall_avg_metrics]).to_csv(OUT_DIR/"overall_avg_across_seeds.csv", index=False)
    pd.DataFrame([overall_ens_stats | overall_ens_metrics]).to_csv(OUT_DIR/"overall_ensemble.csv", index=False)

    print("\n- OVERALL RESULTS ACROSS ALL FOLDS -")
    print_cm_and_metrics("Average Over Seeds (sum across folds)", overall_avg_stats, overall_avg_metrics)
    print_cm_and_metrics("Ensemble (sum across folds)", overall_ens_stats, overall_ens_metrics)
    print("\nSaved all results in:", OUT_DIR)


if __name__ == "__main__":
    compute_fold_confusions(ROOT)


## dist version

In [ ]:

# Freeze the single CV set you selected in Exp-2 
SELECTED_CVSET_ROOT = "./blurclahe_data_folds/blurclahe_inner_seed79"  # median split

OUT_DIR = "results_experiment3_distcorre"

run_deit_on_cvset(
    pipeline_root=SELECTED_CVSET_ROOT,
    out_dir=OUT_DIR,
    seeds=[123],
    epochs=50,
    batch_size=64,
    base_lr=5e-4,         # DeiT paper base LR (scaled by batch/256 inside) :contentReference[oaicite:5]{index=5}
    weight_decay=5e-2,    # common for DeiT/timm recipes :contentReference[oaicite:6]{index=6}
    warmup_epochs=5,
    img_size=224,
    num_workers=0,
    drop_path_rate=0.1,
    device="cuda",
    use_distilled=True
)

Using device: cuda
>> Using DeiT-Small (distilled) with DeiT-Base teacher
[blurclahe_inner_seed79_del fold_4 seed123] Ep 1/50 | TrLoss=0.3055 | ValAUPRC=0.3325 | LR=3.13e-06
[blurclahe_inner_seed79_del fold_4 seed123] Ep 2/50 | TrLoss=0.2782 | ValAUPRC=0.5030 | LR=4.69e-06
[blurclahe_inner_seed79_del fold_4 seed123] Ep 3/50 | TrLoss=0.2563 | ValAUPRC=0.5689 | LR=6.25e-06
[blurclahe_inner_seed79_del fold_4 seed123] Ep 4/50 | TrLoss=0.2496 | ValAUPRC=0.5893 | LR=7.81e-06
[blurclahe_inner_seed79_del fold_4 seed123] Ep 5/50 | TrLoss=0.2435 | ValAUPRC=0.6416 | LR=9.37e-06
[blurclahe_inner_seed79_del fold_4 seed123] Ep 6/50 | TrLoss=0.2384 | ValAUPRC=0.6800 | LR=1.09e-05
[blurclahe_inner_seed79_del fold_4 seed123] Ep 7/50 | TrLoss=0.2397 | ValAUPRC=0.7121 | LR=1.25e-05
[blurclahe_inner_seed79_del fold_4 seed123] Ep 8/50 | TrLoss=0.2343 | ValAUPRC=0.6739 | LR=1.41e-05
[blurclahe_inner_seed79_del fold_4 seed123] Ep 9/50 | TrLoss=0.2321 | ValAUPRC=0.7338 | LR=1.56e-05
[blurclahe_inner_seed79_de

C:\Users\myria\AppData\Local\Temp\ipykernel_3400\3952938368.py:475: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(run_dir / "artifacts" / "b

### compare dist and not dist

In [ ]:
#!/usr/bin/env python3
"""
Compare DeiT-Small (non-distilled vs distilled) on OSDaR23 CV folds.

Outputs:
  comparison_results/
    - results_comparison_summary.csv
    - results_deltas_summary.csv
    - comparison_boxplots.png
    - delta_barplots.png
    - best_val_metric_table.tex
"""



# Paths 
BASELINE_DIR = Path("results_experiment3/DeiT-Small16")
DISTILLED_DIR = Path("results_experiment3_distcorre")
OUT_DIR = Path("comparison_results_dist_nodist")
OUT_DIR.mkdir(exist_ok=True)


# Helpers
def collect_results(exp_dir: Path, tag: str):
    """Collect per-seed val_metrics.json files into one DataFrame."""
    records = []
    for fold in sorted(exp_dir.rglob("fold_*")):
        for seed_dir in sorted(fold.glob("seed*")):
            metrics_path = seed_dir / "val_metrics.json"
            if metrics_path.exists():
                with open(metrics_path, "r") as f:
                    metrics = json.load(f)
                metrics.update({
                    "experiment": tag,
                    "fold": fold.name,
                    "seed": seed_dir.name.replace("seed", ""),
                })
                records.append(metrics)
    return pd.DataFrame(records)


# Load & merge
print("Loading results...")
df_base = collect_results(BASELINE_DIR, "DeiT-Small (non-distilled)")
df_dist = collect_results(DISTILLED_DIR, "DeiT-Small (distilled)")
df_all = pd.concat([df_base, df_dist], ignore_index=True)

metrics_cols = ["accuracy", "balanced_accuracy", "auroc", "auprc", "precision", "recall", "f1"]
df_all = df_all[["experiment", "fold", "seed"] + metrics_cols]

# Aggregated summary
summary = (
    df_all.groupby("experiment")[metrics_cols]
    .agg(["mean", "std"])
    .round(4)
)
summary.columns = ["_".join(col) for col in summary.columns]
summary.to_csv(OUT_DIR / "results_comparison_summary.csv")

# LaTeX table
latex_table = summary.copy()
latex_table.index.name = "Model"
latex_str = latex_table.to_latex(
    caption="Comparison of DeiT-Small with and without distillation on OSDaR23 dataset.",
    label="tab:deit_comparison",
    float_format="%.4f"
)
with open(OUT_DIR / "best_val_metric_table.tex", "w") as f:
    f.write(latex_str)

# diff per fold (distilled − baseline)
merged = pd.merge(
    df_base, df_dist,
    on=["fold", "seed"],
    suffixes=("_base", "_dist")
)
for m in metrics_cols:
    merged[f"Δ_{m}"] = merged[f"{m}_dist"] - merged[f"{m}_base"]

delta_cols = [f"Δ_{m}" for m in metrics_cols]
delta_summary = merged.groupby("fold")[delta_cols].mean().round(4)
delta_summary.loc["mean_over_folds"] = delta_summary.mean()
delta_summary.to_csv(OUT_DIR / "results_deltas_summary.csv")

# Visualization
sns.set(style="whitegrid", font_scale=1.2)

# - Boxplots (overall performance)
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()
metrics_to_plot = ["accuracy", "balanced_accuracy", "auroc", "auprc", "precision", "f1"]

for ax, metric in zip(axes, metrics_to_plot):
    sns.boxplot(data=df_all, x="experiment", y=metric, palette="Set2", ax=ax)
    ax.set_title(metric.replace("_", " ").title())
    ax.set_xlabel("")
    ax.set_ylabel(metric)
    ax.tick_params(axis="x", rotation=15)

fig.suptitle("DeiT-Small vs DeiT-Small (Distilled) — OSDaR23", fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / "comparison_boxplots.png", dpi=300)
plt.close()

# - diff per fold barplots
melted = merged.melt(
    id_vars=["fold", "seed"], value_vars=delta_cols,
    var_name="metric", value_name="delta"
)
melted["metric"] = melted["metric"].str.replace("Δ_", "")

plt.figure(figsize=(12, 6))
sns.barplot(data=melted, x="fold", y="delta", hue="metric", ci=None, palette="tab10")
plt.axhline(0, color="black", lw=1)
plt.title("Per-fold Δ (Distilled − Non-distilled) across metrics")
plt.ylabel("Performance gain")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.savefig(OUT_DIR / "delta_barplots.png", dpi=300)
plt.close()

# Print summaries
print("\n- Aggregated summary (mean ± std) -")
print(summary)
print("\n-Mean Δ across folds (Distilled − Non-distilled) -")
print(delta_summary.loc["mean_over_folds"])
print("\nSaved to:", OUT_DIR)


# diff Heatmap per fold

# Remove mean row for plotting
delta_plot_df = delta_summary.drop(index="mean_over_folds", errors="ignore")

plt.figure(figsize=(10, 5))
sns.heatmap(
    delta_plot_df,
    annot=True,
    fmt=".3f",
    cmap=sns.diverging_palette(240, 10, as_cmap=True),
    center=0,
    cbar_kws={"label": "Δ (Distilled − Non-distilled)"}
)
plt.title("Per-fold Performance Δ — DeiT-Small Distilled vs Non-distilled")
plt.ylabel("Fold")
plt.xlabel("Metric")
plt.tight_layout()
plt.savefig(OUT_DIR / "delta_heatmap.png", dpi=300)
plt.close()
print("Saved heatmap:", OUT_DIR / "delta_heatmap.png")



Loading results...


C:\Users\myria\AppData\Local\Temp\ipykernel_3400\2574695587.py:111: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df_all, x="experiment", y=metric, palette="Set2", ax=ax)
C:\Users\myria\AppData\Local\Temp\ipykernel_3400\2574695587.py:111: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df_all, x="experiment", y=metric, palette="Set2", ax=ax)
C:\Users\myria\AppData\Local\Temp\ipykernel_3400\2574695587.py:111: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df_all, x="experiment", y=metric, palette="Set2", ax=ax)
C:\Users\myria\AppD


===== Aggregated summary (mean ± std) =====
                            accuracy_mean  accuracy_std  \
experiment                                                
DeiT-Small (distilled)             0.9394        0.0253   
DeiT-Small (non-distilled)         0.9441        0.0285   

                            balanced_accuracy_mean  balanced_accuracy_std  \
experiment                                                                  
DeiT-Small (distilled)                      0.9100                 0.0749   
DeiT-Small (non-distilled)                  0.9287                 0.0560   

                            auroc_mean  auroc_std  auprc_mean  auprc_std  \
experiment                                                                 
DeiT-Small (distilled)          0.9553     0.0340      0.8853     0.1004   
DeiT-Small (non-distilled)      0.9706     0.0324      0.9107     0.0960   

                            precision_mean  precision_std  recall_mean  \
experiment                    

# PLOTS

In [ ]:
# Set style for quality plots
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context("paper", font_scale=1.3)
sns.set_palette("deep")
plt.rcParams.update({
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'font.size': 15,
    'axes.labelsize': 15,
    'axes.titlesize': 25,
    'xtick.labelsize': 15,
    'ytick.labelsize': 15,
    'legend.fontsize': 15,
    'legend.framealpha': 0.95,
    'legend.edgecolor': '0.8',
    'legend.fancybox': True,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': '--',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.linewidth': 1.2,
    'lines.linewidth': 2.5,
    'lines.markersize': 8,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.1
})


#CONFIGURATION
BASE_DIR = Path("./results_experiment3final")
OUTPUT_DIR = Path("./results_experiment3_analysis")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

SEEDS = [0, 42, 123]
FOLDS = [0, 1, 2, 3, 4]

MODELS = {
    'ResNet-18': {
        'dir': BASE_DIR / "Resnet-18/blurclahe_inner_seed79",
        'supervised': True,
        'label_convention': 'class',
        'color': '#D62728',  #  red
        'marker': 'o'
    },
    'DeiT-Small': {
        'dir': BASE_DIR / "DeiT-Small16/blurclahe_inner_seed79",
        'supervised': True,
        'label_convention': 'class',
        'color': '#1F77B4',  #  blue
        'marker': 's'
    },
    'PatchCore': {
        'dir': BASE_DIR / "Patchcore",
        'supervised': False,
        'label_convention': 'anomaly',
        'color': '#2CA02C',  #  green
        'marker': '^'
    }
}


# DATA LOADING FUNCTIONS

def load_supervised_results(model_name, model_config):
    """Load results for ResNet-18 or DeiT-Small."""
    results = {
        'training_history': [],
        'val_metrics': [],
        'val_predictions': []
    }
    
    for fold in FOLDS:
        for seed in SEEDS:
            fold_dir = model_config['dir'] / f"fold_{fold}" / f"seed{seed}"
            
            if not fold_dir.exists():
                continue
            
            # Training history
            history_file = fold_dir / "training_history.csv"
            if history_file.exists():
                history = pd.read_csv(history_file)
                history['model'] = model_name
                history['fold'] = fold
                history['seed'] = seed
                results['training_history'].append(history)
            
            # Validation metrics
            if history_file.exists():
                best_epoch = history['val_auprc'].idxmax()
                best_metrics = history.iloc[best_epoch].to_dict()
                best_metrics['model'] = model_name
                best_metrics['fold'] = fold
                best_metrics['seed'] = seed
                results['val_metrics'].append(best_metrics)
            
            # Validation predictions
            val_pred_file = fold_dir / "val_predictions.csv"
            if val_pred_file.exists():
                val_preds = pd.read_csv(val_pred_file)
                val_preds['model'] = model_name
                val_preds['fold'] = fold
                val_preds['seed'] = seed
                results['val_predictions'].append(val_preds)
    
    # Concatenate results
    if results['training_history']:
        results['training_history'] = pd.concat(results['training_history'], ignore_index=True)
    else:
        results['training_history'] = pd.DataFrame()
        
    if results['val_metrics']:
        results['val_metrics'] = pd.DataFrame(results['val_metrics'])
    else:
        results['val_metrics'] = pd.DataFrame()
        
    if results['val_predictions']:
        results['val_predictions'] = pd.concat(results['val_predictions'], ignore_index=True)
    else:
        results['val_predictions'] = pd.DataFrame()
    
    return results


def load_patchcore_results(model_config):
    """Load PatchCore results."""
    
    
    results = {
        'val_metrics': [],
        'val_predictions': []
    }
    
    for seed in SEEDS:
        seed_dir = model_config['dir'] / f"seed{seed}_IM224_WR50_L2-3_P01_D1024-1024_PS-3_AN-1_S0"
        
        if not seed_dir.exists():
            continue
        
        per_image_dir = seed_dir / "per_image"
        if not per_image_dir.exists():
            continue
        
        for fold in FOLDS:
            fold_dir = per_image_dir / f"mvtec_fold_{fold}79"
            scores_file = fold_dir / "image_scores.csv"
            
            if not scores_file.exists():
                continue
            
            preds = pd.read_csv(scores_file)
            preds['filename'] = preds['filepath'].apply(lambda x: Path(x).name)
            preds['model'] = 'PatchCore'
            preds['fold'] = fold
            preds['seed'] = seed
            
            # Convert to our convention
            preds['y_true'] = 1 - preds['is_defect'].astype(int)
            preds['y_pred'] = 1 - preds['predicted_label'].astype(int)
            preds['p_class_0'] = preds['score']
            preds['p_class_1'] = 1 - preds['score']
            
            results['val_predictions'].append(preds)
            
            # Compute metrics
            y_true_defect = preds['is_defect'].values
            y_pred_defect = preds['predicted_label'].values
            y_score = preds['score'].values
            
            if len(np.unique(y_true_defect)) < 2:
                continue
            
            prec, rec, f1, _ = precision_recall_fscore_support(
                y_true_defect, y_pred_defect, pos_label=1, average='binary', zero_division=0
            )
            
            try:
                auroc = roc_auc_score(y_true_defect, y_score)
                auprc = average_precision_score(y_true_defect, y_score)
            except:
                auroc, auprc = np.nan, np.nan
            
            y_true_class = preds['y_true'].values
            y_pred_class = preds['y_pred'].values
            
            metrics = {
                'model': 'PatchCore',
                'fold': fold,
                'seed': seed,
                'val_auroc': float(auroc),
                'val_auprc': float(auprc),
                'val_precision': float(prec),
                'val_recall': float(rec),
                'val_f1': float(f1),
                'val_accuracy': accuracy_score(y_true_class, y_pred_class),
                'val_balanced_accuracy': balanced_accuracy_score(y_true_class, y_pred_class),
            }
            results['val_metrics'].append(metrics)
    
    if results['val_metrics']:
        results['val_metrics'] = pd.DataFrame(results['val_metrics'])
    else:
        results['val_metrics'] = pd.DataFrame()
        
    if results['val_predictions']:
        results['val_predictions'] = pd.concat(results['val_predictions'], ignore_index=True)
    else:
        results['val_predictions'] = pd.DataFrame()
    
    return results



# MAIN 

def main():
    print("EXPERIMENT 3: MODEL COMPARISON ANALYSIS")
    
    # Load all data
    print("\nLoading all model results...")
    all_results = {}
    
    for model_name, config in MODELS.items():
        if config['supervised']:
            all_results[model_name] = load_supervised_results(model_name, config)
        else:
            all_results[model_name] = load_patchcore_results(config)
    
    # Combine all metrics
    all_metrics = pd.concat([
        all_results[model]['val_metrics'] 
        for model in MODELS.keys() 
        if len(all_results[model]['val_metrics']) > 0
    ], ignore_index=True)
    
    print(f"\nLoaded {len(all_metrics)} total fold/seed combinations")
    print(all_metrics.groupby('model').size())
    
    # PLOT 1: PRECISION-RECALL CURVES
    
    print("Generating Precision-Recall Curves...")
    
    fig, ax = plt.subplots(figsize=(10, 7))
    
    for model_name in MODELS.keys():
        preds = all_results[model_name]['val_predictions']
        
        if len(preds) == 0:
            print(f" No predictions for {model_name}")
            continue
        
        pr_curves = []
        auprc_scores = []
        
        for (fold, seed), group in preds.groupby(['fold', 'seed']):
            y_true = group['y_true'].values
            
            if 'p_class_0' not in group.columns:
                continue
            
            y_score = group['p_class_0'].values
            y_true_defect = 1 - y_true
            
            precision, recall, _ = precision_recall_curve(y_true_defect, y_score)
            pr_auc = auc(recall, precision)
            
            pr_curves.append(np.column_stack([recall, precision]))
            auprc_scores.append(pr_auc)
        
        if len(pr_curves) == 0:
            continue
        
        # Interpolate to common recall values
        recall_common = np.linspace(0, 1, 100)
        precision_interp = []
        
        for pr_curve in pr_curves:
            recall = pr_curve[:, 0]
            precision = pr_curve[:, 1]
            
            idx = np.argsort(recall)
            recall = recall[idx]
            precision = precision[idx]
            
            prec_interp = np.interp(recall_common, recall, precision, 
                                   left=precision[0], right=precision[-1])
            precision_interp.append(prec_interp)
        
        precision_interp = np.array(precision_interp)
        precision_mean = np.mean(precision_interp, axis=0)
        precision_std = np.std(precision_interp, axis=0)
        
        mean_auprc = np.mean(auprc_scores)
        std_auprc = np.std(auprc_scores)
        
        color = MODELS[model_name]['color']
        ax.plot(recall_common, precision_mean, 
                color=color, linewidth=3,
                label=f'{model_name} (AUPRC = {mean_auprc:.3f} ± {std_auprc:.3f})',
                zorder=3)
        
        ax.fill_between(recall_common, 
                        precision_mean - precision_std,
                        precision_mean + precision_std,
                        color=color, alpha=0.15, zorder=2)
    
    # Add baseline (random classifier)
    ax.plot([0, 1], [0.5, 0.5], 'k--', linewidth=1.5, 
            label='Random Classifier', alpha=0.5, zorder=1)
    
    ax.set_xlabel('Recall', fontsize=15)
    ax.set_ylabel('Precision', fontsize=15)
    ax.set_title('Precision-Recall Curves', fontsize=25)
    ax.legend(loc='lower left', framealpha=0.95, fontsize=15)
    ax.set_xlim([-0.02, 1.02])
    ax.set_ylim([-0.02, 1.02])
    ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.8)
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'exp3_precision_recall_curves.png', dpi=300, bbox_inches='tight')
    print(f"Saved: exp3_precision_recall_curves.png")
    plt.close()
    
    
    
    # PLOT 2: VALIDATION AUPRC CURVES (All models on same plot)
    
    print("\nGenerating Validation AUPRC Curves...")
    
    supervised_models = [m for m in MODELS.keys() if MODELS[m]['supervised']]
    
    if len(supervised_models) > 0:
        fig, ax = plt.subplots(figsize=(10, 7))
        
        for model_name in supervised_models:
            history = all_results[model_name]['training_history']
            
            if len(history) == 0:
                continue
            
            epoch_stats = history.groupby('epoch').agg({
                'val_auprc': ['mean', 'std']
            }).reset_index()
            
            epochs = epoch_stats['epoch'].values
            color = MODELS[model_name]['color']
            
            ax.plot(epochs, epoch_stats[('val_auprc', 'mean')], 
                    color=color, linewidth=3, label=model_name, zorder=3)
            ax.fill_between(epochs,
                           epoch_stats[('val_auprc', 'mean')] - epoch_stats[('val_auprc', 'std')],
                           epoch_stats[('val_auprc', 'mean')] + epoch_stats[('val_auprc', 'std')],
                           color=color, alpha=0.15, zorder=2)
        
        ax.set_xlabel('Epoch', fontsize=15)
        ax.set_ylabel('Validation AUPRC', fontsize=15)
        ax.set_title('Validation AUPRC over Training Epochs', fontsize=25)
        ax.legend(loc='lower right', framealpha=0.95, fontsize=15)
        ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.8)
        ax.set_ylim([0, 1.02])
        ax.set_xlim([0, epochs[-1]])
        
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / 'exp3_val_auprc_curves.png', dpi=300, bbox_inches='tight')
        print(f"Saved: exp3_val_auprc_curves.png")
        plt.close()
    

    # PLOT 3: CONFUSION MATRICES (Normalized)
    
    print("\nGenerating Confusion Matrices...")
    
    
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    for idx, model_name in enumerate(MODELS.keys()):
        preds = all_results[model_name]['val_predictions']
        
        if len(preds) == 0:
            continue
        
        y_true = preds['y_true'].values
        y_pred = preds['y_pred'].values
        
        cm = confusion_matrix(y_true, y_pred)
        cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        
        ax = axes[idx]
        im = ax.imshow(cm_normalized, interpolation='nearest', 
                      cmap='Blues', vmin=0, vmax=1)
        ax.set_title(model_name, fontweight='bold', fontsize=14, pad=10)
        
        cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label('Proportion', rotation=270, labelpad=20, 
                      fontweight='bold', fontsize=11)
        cbar.ax.tick_params(labelsize=10)
        
        thresh = cm_normalized.max() / 2.
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(j, i, f'{cm[i, j]}\n({cm_normalized[i, j]:.2%})',
                       ha="center", va="center",
                       color="white" if cm_normalized[i, j] > thresh else "black",
                       fontsize=11, fontweight='bold')
        
        ax.set_ylabel('True Label', fontweight='bold', fontsize=12)
        ax.set_xlabel('Predicted Label', fontweight='bold', fontsize=12)
        ax.set_xticks([0, 1])
        ax.set_yticks([0, 1])
        ax.set_xticklabels(['Defective', 'Good'], fontsize=11)
        ax.set_yticklabels(['Defective', 'Good'], fontsize=11)
    
    plt.subplots_adjust(wspace=0.4)
    plt.savefig(OUTPUT_DIR / 'exp3_confusion_matrices.png', dpi=300, bbox_inches='tight')
    print(f"Saved: exp3_confusion_matrices.png")
    plt.close()
    

    # TABLE: SUMMARY STATISTICS
    
    print("\nGenerating Summary Statistics Table...")
    
    metric_cols = ['val_auprc', 'val_auroc', 'val_f1', 'val_precision', 'val_recall', 'val_accuracy']
    metric_names = ['AUPRC', 'AUROC', 'F1-Score', 'Precision', 'Recall', 'Accuracy']
    
    summary_table = []
    
    for model in MODELS.keys():
        model_metrics = all_metrics[all_metrics['model'] == model]
        row = {'Model': model}
        
        for col, name in zip(metric_cols, metric_names):
            mean_val = model_metrics[col].mean()
            std_val = model_metrics[col].std()
            row[name] = f"{mean_val:.3f} ± {std_val:.3f}"
        
        summary_table.append(row)
    
    summary_df = pd.DataFrame(summary_table)
    
    # Find best performers
    best_per_metric = {}
    for col, name in zip(metric_cols, metric_names):
        best_model = all_metrics.groupby('model')[col].mean().idxmax()
        best_per_metric[name] = best_model
    
    # Save to CSV
    summary_df.to_csv(OUTPUT_DIR / 'exp3_summary_table.csv', index=False)
    print(f"Saved: exp3_summary_table.csv")
    
    # Save to LaTeX
    latex_table = summary_df.to_latex(index=False, escape=False)
    with open(OUTPUT_DIR / 'exp3_summary_table.tex', 'w') as f:
        f.write(latex_table)
    print(f"Saved: exp3_summary_table.tex")
    
    # Print to console

    print("SUMMARY STATISTICS TABLE")

    print(summary_df.to_string(index=False))
    print("\nBest performers per metric:")
    for name, model in best_per_metric.items():
        print(f"  {name}: {model}")
    
    
    print("All visualizations generated successfully!")
    print(f"Output directory: {OUTPUT_DIR}")
  


if __name__ == "__main__":
    main()

EXPERIMENT 3: MODEL COMPARISON ANALYSIS

Loading all model results...

Loaded 45 total fold/seed combinations
model
DeiT-Small    15
PatchCore     15
ResNet-18     15
dtype: int64
Generating Precision-Recall Curves...
Saved: exp3_precision_recall_curves.png

Generating Validation AUPRC Curves...
Saved: exp3_val_auprc_curves.png

Generating Confusion Matrices...
Saved: exp3_confusion_matrices.png

Generating Summary Statistics Table...
Saved: exp3_summary_table.csv
Saved: exp3_summary_table.tex
SUMMARY STATISTICS TABLE
     Model         AUPRC         AUROC      F1-Score     Precision        Recall      Accuracy
 ResNet-18 0.900 ± 0.120 0.965 ± 0.040 0.773 ± 0.115 0.809 ± 0.158 0.788 ± 0.192 0.900 ± 0.040
DeiT-Small 0.937 ± 0.073 0.977 ± 0.031 0.814 ± 0.121 0.823 ± 0.161 0.859 ± 0.189 0.922 ± 0.043
 PatchCore 0.608 ± 0.185 0.834 ± 0.094 0.680 ± 0.117 0.591 ± 0.153 0.839 ± 0.076 0.801 ± 0.086

Best performers per metric:
  AUPRC: DeiT-Small
  AUROC: DeiT-Small
  F1-Score: DeiT-Small
  Pr